# 문제 2-1 : 콤마 구분 리스트 파서 활용

관심 분야를 입력하면 **한국의 유명한 장소나 활동 5가지**를 리스트로 출력합니다.

- `CommaSeparatedListOutputParser` 사용
- 사용자 입력을 받아 동적으로 처리

### 공통 설정: API Key 로드 + LLM 생성 함수

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True, **kwargs) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
        **kwargs,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

# 콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

# 출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [4]:
# 프롬프트 템플릿 설정 (형식 지침은 partial_variables 로 미리 주입)
prompt = PromptTemplate(
    template="당신은 한국 여행·문화 전문가입니다.\n"
             "'{subject}' 분야와 관련된 한국의 유명한 장소나 활동 5가지를 추천하세요.\n"
             "각 항목은 '지역/장소 + 대상' 형태의 짧은 한국어 명사구로 작성하세요. (예: 전주 비빔밥)\n"
             "{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

llm = get_llm(temperature=0.7)

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | llm | output_parser

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [5]:
# 예시 입력: "음식"
result = chain.invoke({"subject": "음식"})
print(type(result))  # list
print(result)

<class 'list'>
['부산 밀면', '제주 흑돼지', '전주 비빔밥', '경주 한정식', '강릉 커피']


In [6]:
# 사용자 입력을 받아 동적으로 처리 (빈 값이면 "스포츠")
subject = input("관심 있는 분야를 입력하세요 (예: 음식, 스포츠, 영화): ").strip() or "스포츠"
result = chain.invoke({"subject": subject})

print(f"[{subject}] 관련 한국 추천 5가지")
print(result)
for i, item in enumerate(result, 1):
    print(f"{i}. {item}")

[음식] 관련 한국 추천 5가지
['전주 비빔밥', '부산 밀면', '제주 갈치조림', '강릉 초당 순두부', '경주 한우']
1. 전주 비빔밥
2. 부산 밀면
3. 제주 갈치조림
4. 강릉 초당 순두부
5. 경주 한우


In [7]:
# 여러 분야 한 번에 테스트
subjects = ["영화", "역사", "자연"]
for subject, items in zip(subjects, chain.batch([{"subject": s} for s in subjects])):
    print(f"{subject}: {items}")

영화: ['부산 영화의 전당', '경기 파주 헤이리 영화관', '제주 섭지코지 촬영지', '서울 남산 드라마촬영지', '강원 평창 영화제']
역사: ['경주 불국사', '수원 화성', '서울 경복궁', '공주 무령왕릉', '부산 동래읍성']
자연: ['제주 올레길', '강원도 설악산 등산', '경상북도 울릉도 트레킹', '전라남도 다도해 카약', '충청북도 단양 구담폭포 탐방']
